In [1]:
import yfinance as yf
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


In [2]:
STOCKS = ["AAPL", "MSFT", "JPM", "TSLA", "GOOGL"]
ETFS = ["SPY", "QQQ", "VTI", "IWM", "DIA"]
BONDS = ["AGG", "TLT", "LQD", "BND", "SHY"]
TICKERS = STOCKS + ETFS + BONDS

# A bond fund and an equity fund both report quoteType == "ETF" in yfinance -
# category text is what actually tells them apart.
BOND_CATEGORY_KEYWORDS = [
    "bond", "government", "treasury", "muni", "inflation-protected",
    "debt", "bank loan", "target maturity",
]

ASSET_CLASS_LABELS = {"stock": "Equity", "etf": "ETF", "bond": "Bond"}


In [3]:
def fetch_yfinance_data(tickers):
    data = {}
    for symbol in tickers:
        info = yf.Ticker(symbol).info
        if not info or info.get("quoteType") is None:
            print(f"✗ No data for {symbol}")
            continue
        data[symbol] = info
        print(f"✓ Fetched {symbol}: quoteType={info.get('quoteType')}")
    return data

yf_data = fetch_yfinance_data(TICKERS)


✓ Fetched AAPL: quoteType=EQUITY
✓ Fetched MSFT: quoteType=EQUITY
✓ Fetched JPM: quoteType=EQUITY
✓ Fetched TSLA: quoteType=EQUITY
✓ Fetched GOOGL: quoteType=EQUITY
✓ Fetched SPY: quoteType=ETF
✓ Fetched QQQ: quoteType=ETF
✓ Fetched VTI: quoteType=ETF
✓ Fetched IWM: quoteType=ETF
✓ Fetched DIA: quoteType=ETF
✓ Fetched AGG: quoteType=ETF
✓ Fetched TLT: quoteType=ETF
✓ Fetched LQD: quoteType=ETF
✓ Fetched BND: quoteType=ETF
✓ Fetched SHY: quoteType=ETF


In [4]:
def classify_instrument(info):
    quote_type = info.get("quoteType")
    category = (info.get("category") or "").lower()

    if quote_type == "EQUITY":
        return "stock"
    if quote_type == "ETF":
        return "bond" if any(kw in category for kw in BOND_CATEGORY_KEYWORDS) else "etf"
    return "unknown"

for symbol, info in yf_data.items():
    print(f"{symbol:6s} category={info.get('category')!r:30s} -> {classify_instrument(info)}")


AAPL   category=None                           -> stock
MSFT   category=None                           -> stock
JPM    category=None                           -> stock
TSLA   category=None                           -> stock
GOOGL  category=None                           -> stock
SPY    category='Large Blend'                  -> etf
QQQ    category='Large Growth'                 -> etf
VTI    category='Large Blend'                  -> etf
IWM    category='Small Blend'                  -> etf
DIA    category='Large Value'                  -> etf
AGG    category='Intermediate Core Bond'       -> bond
TLT    category='Long Government'              -> bond
LQD    category='Corporate Bond'               -> bond
BND    category='Intermediate Core Bond'       -> bond
SHY    category='Short Government'             -> bond


In [5]:
def build_stock_row(symbol, info):
    return {
        "symbol": symbol,
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "country": info.get("country"),
        "market_cap": info.get("marketCap"),
        "shares_outstanding": info.get("sharesOutstanding"),
        "full_time_employees": info.get("fullTimeEmployees"),
        "beta": info.get("beta"),
        "trailing_pe": info.get("trailingPE"),
        "forward_pe": info.get("forwardPE"),
        "trailing_eps": info.get("trailingEps"),
        "dividend_rate": info.get("dividendRate"),
        "payout_ratio": info.get("payoutRatio"),
        "price_to_book": info.get("priceToBook"),
        "return_on_equity": info.get("returnOnEquity"),
        "total_revenue": info.get("totalRevenue"),
        "website": info.get("website"),
    }


def build_fund_row(symbol, info):
    # shared by etfs and bonds - both are fund-shaped in yfinance
    return {
        "symbol": symbol,
        "category": info.get("category"),
        "fund_family": info.get("fundFamily"),
        "legal_type": info.get("legalType"),
        "net_expense_ratio": info.get("netExpenseRatio"),
        "nav_price": info.get("navPrice"),
        "total_assets": info.get("totalAssets"),
        "net_assets": info.get("netAssets"),
        "ytd_return": info.get("ytdReturn"),
        "three_year_avg_return": info.get("threeYearAverageReturn"),
        "five_year_avg_return": info.get("fiveYearAverageReturn"),
        "beta_3_year": info.get("beta3Year"),
        "distribution_yield": info.get("yield"),
    }


In [6]:
def clean(df):
    # pandas turns missing values into NaN - swap back to None
    return df.astype(object).where(pd.notnull(df), None)


instrument_rows, stock_rows, etf_rows, bond_rows = [], [], [], []

for symbol, info in yf_data.items():
    asset_class = classify_instrument(info)
    if asset_class == "unknown":
        continue

    instrument_rows.append({
        "symbol": symbol,
        "name": info.get("shortName") or info.get("longName") or symbol,
        "asset_class": ASSET_CLASS_LABELS[asset_class],
        "currency": info.get("currency"),
        "exchange": info.get("fullExchangeName"),
    })

    if asset_class == "stock":
        stock_rows.append(build_stock_row(symbol, info))
    elif asset_class == "etf":
        etf_rows.append(build_fund_row(symbol, info))
    elif asset_class == "bond":
        bond_rows.append(build_fund_row(symbol, info))

instruments_df = clean(pd.DataFrame(instrument_rows))
stocks_df = clean(pd.DataFrame(stock_rows))
etfs_df = clean(pd.DataFrame(etf_rows))
bonds_df = clean(pd.DataFrame(bond_rows))

print(f"instruments: {len(instruments_df)}  stocks: {len(stocks_df)}  "
      f"etfs: {len(etfs_df)}  bonds: {len(bonds_df)}")


instruments: 15  stocks: 5  etfs: 5  bonds: 5


In [7]:
instruments_df

,symbol,name,asset_class,currency,exchange
0,AAPL,Apple Inc.,Equity,USD,NasdaqGS
1,MSFT,Microsoft Corporation,Equity,USD,NasdaqGS
2,JPM,JP Morgan Chase & Co.,Equity,USD,NYSE
3,TSLA,"Tesla, Inc.",Equity,USD,NasdaqGS
4,GOOGL,Alphabet Inc.,Equity,USD,NasdaqGS
5,SPY,State Street SPDR S&P 500 ETF T,ETF,USD,NYSEArca
6,QQQ,"Invesco QQQ Trust, Series 1",ETF,USD,NasdaqGM
7,VTI,Vanguard Morningstar Total Stoc,ETF,USD,NYSEArca
8,IWM,iShares Russell 2000 Index Fund,ETF,USD,NYSEArca
9,DIA,State Street SPDR Dow Jones Ind,ETF,USD,NYSEArca


In [8]:
stocks_df

,symbol,sector,industry,country,market_cap,shares_outstanding,full_time_employees,beta,trailing_pe,forward_pe,trailing_eps,dividend_rate,payout_ratio,price_to_book,return_on_equity,total_revenue,website
0,AAPL,Technology,Consumer Electronics,United States,4849207869440,14594180000,150000,1.085,38.148106,34.673397,8.71,1.08,0.1204,45.14538,1.48751,466822987776,https://www.apple.com
1,MSFT,Technology,Software - Infrastructure,United States,3680323239936,7425545491,223000,1.108,27.62709,21.025343,17.94,3.64,0.1983,8.320827,0.34039,331839012864,https://www.microsoft.com
2,JPM,Financial Services,Banks - Diversified,United States,946925731840,2658186195,320560,0.975,15.269182,14.256049,23.33,6.0,0.2571,2.67828,0.17789,186328006656,https://www.jpmorganchase.com
3,TSLA,Consumer Cyclical,Auto Manufacturers,United States,1443322658816,3949547394,134785,1.845,332.21817,169.29648,1.1,None,0.0,16.614685,0.04667,103619002368,https://www.tesla.com
4,GOOGL,Communication Services,Internet Content & Information,United States,4139833098240,5867155790,198933,1.225,16.975927,22.758093,19.94,0.88,0.0426,6.650817,0.48676,445865984000,https://abc.xyz


In [9]:
etfs_df

,symbol,category,fund_family,legal_type,net_expense_ratio,nav_price,total_assets,net_assets,ytd_return,three_year_avg_return,five_year_avg_return,beta_3_year,distribution_yield
0,SPY,Large Blend,State Street Investment Management,Exchange Traded Fund,0.0945,764.3512,811937038336,811937040000.0,13.07293,0.209433,0.129232,1.0,0.0098
1,QQQ,Large Growth,Invesco,Exchange Traded Fund,0.18,714.99,488981004288,488981004000.0,16.99566,0.244914,0.143591,1.26,0.0042
2,VTI,Large Blend,Vanguard,Exchange Traded Fund,0.03,376.31,2343736705024,2343736710000.0,13.44794,0.206659,0.119023,1.02,0.0103
3,IWM,Small Blend,iShares,Exchange Traded Fund,0.19,289.00253,80460996608,80460997000.0,19.90827,0.175349,0.067708,1.24,0.009
4,DIA,Large Value,State Street Investment Management,Exchange Traded Fund,0.16,525.90234,45455503360,45455503000.0,11.54551,0.166838,0.105825,0.83,0.0138


In [10]:
bonds_df

,symbol,category,fund_family,legal_type,net_expense_ratio,nav_price,total_assets,net_assets,ytd_return,three_year_avg_return,five_year_avg_return,beta_3_year,distribution_yield
0,AGG,Intermediate Core Bond,iShares,Exchange Traded Fund,0.03,95.93898,138317873152,138317873000.0,-0.22523,0.039952,-0.005025,0.99,0.0405
1,TLT,Long Government,iShares,Exchange Traded Fund,0.15,80.81934,47046328320,47046328000.0,-2.88047,-0.006736,-0.082564,2.39,0.0473
2,LQD,Corporate Bond,iShares,Exchange Traded Fund,0.14,104.33994,32042811392,32042811400.0,-1.1585,0.045205,-0.011919,1.35,0.0467
3,BND,Intermediate Core Bond,Vanguard,Exchange Traded Fund,0.03,71.19,398834597888,398834598000.0,-0.18909,0.040056,-0.005161,0.98,0.0404
4,SHY,Short Government,iShares,Exchange Traded Fund,0.15,81.33795,25913065472,25913065500.0,0.94928,0.040072,0.017572,0.22,0.0363
